In [1]:
import pandas as pd

In [2]:
crosswalk = pd.read_csv('crosswalk.tsv', sep='\t')

In [3]:
crosswalk

,FSTATE,FCOUNTY,FPLACE,FIPS_ST,FIPS_COUNTY,FIPS,ORI9,ORI7,NAME,UA,...,INTPTLONG,CONGDIST1,CONGDIST2_18,DISTNAME,SOURCE_CSLLEA2008,SOURCE_UCR2010,SOURCE_UCR2011,SOURCE_UCR2012,SOURCE_NCIC2012,SOURCE_VENDOR
0,1,1,3220,1,1,1001,AL0040200,AL00402,AUTAUGAVILLE POLICE DEPARTMENT,-1,...,-86.644490,2,,Middle District of Alabama,1,1,1,1,1,1
1,1,1,62328,1,1,1001,AL0040100,AL00401,PRATTVILLE POLICE DEPARTMENT,58600,...,-86.644490,2,,Middle District of Alabama,1,1,1,1,1,1
2,1,1,62328,1,1,1001,AL0040300,-1,PRATTVILLE FIRE DEPT ARSON INVESTIGATION BRANCH,58600,...,-86.644490,2,,Middle District of Alabama,0,0,0,0,1,0
3,1,1,99001,1,1,1001,AL0040000,AL00400,AUTAUGA COUNTY SHERIFF'S OFFICE,-1,...,-86.644490,2,,Middle District of Alabama,1,1,1,1,1,1
4,1,3,4660,1,3,1003,AL0051100,AL00511,FAULKNER STATE COMMUNITY COLLEGE POLICE DEPT,5923,...,-87.746067,1,,Southern District of Alabama,1,0,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36485,56,43,99043,56,43,56043,WY0220000,WY02200,WASHAKIE COUNTY SHERIFF'S OFFICE,-1,...,-107.669052,1,,District of Wyoming,1,1,1,1,1,1
36486,56,45,56215,56,45,56045,WY0230100,WY02301,NEWCASTLE POLICE DEPT,62191,...,-104.570020,1,,District of Wyoming,1,1,1,1,1,1
36487,56,45,56215,56,45,56045,WYWHP1200,-1,SHP DIV L NEWCASTLE,62191,...,-104.570020,1,,District of Wyoming,0,0,0,0,1,0
36488,56,45,79125,56,45,56045,WY0230200,WY02302,UPTON POLICE DEPARTMENT,-1,...,-104.570020,1,,District of Wyoming,1,1,1,1,1,1


In [4]:
crosswalk.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36490 entries, 0 to 36489
Data columns (total 46 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   FSTATE             36490 non-null  int64  
 1   FCOUNTY            36490 non-null  int64  
 2   FPLACE             36490 non-null  int64  
 3   FIPS_ST            36490 non-null  int64  
 4   FIPS_COUNTY        36490 non-null  int64  
 5   FIPS               36490 non-null  int64  
 6   ORI9               36490 non-null  object 
 7   ORI7               36490 non-null  object 
 8   NAME               36490 non-null  object 
 9   UA                 36490 non-null  int64  
 10  STATENAME          36490 non-null  object 
 11  COUNTYNAME         36490 non-null  object 
 12  UANAME             36490 non-null  object 
 13  PARTOF             36490 non-null  int64  
 14  AGCYTYPE           36490 non-null  int64  
 15  SUBTYPE1           36490 non-null  int64  
 16  SUBTYPE2           364

## We only keep the ORI, County FIPS, Place FIPS, statename, countyname, local government name, local government population and urban area code.

In [5]:
# 1. Keep only the columns we care about 
cols_keep = [
    'ORI9',        # agency ID
    'FIPS',        # county FIPS
    'FPLACE',      # place FIPS
    'STATENAME',   # state name
    'COUNTYNAME',  # county name
    'LG_NAME',     # place/local government name
    'LG_POPULATION',
    'UA'           # urban area code
]

crosswalk = crosswalk[cols_keep].copy()

In [6]:
# 2. Rename columns to friendlier names
crosswalk = crosswalk.rename(columns={
    'ORI9': 'ori',
    'FIPS': 'county_fips',
    'FPLACE': 'place_fips',
    'STATENAME': 'state_name',
    'COUNTYNAME': 'county_name',
    'LG_NAME': 'place_name',
    'LG_POPULATION': 'lg_population',
    'UA': 'ua'
})

In [7]:
crosswalk.isnull().sum()

ori              0
county_fips      0
place_fips       0
state_name       0
county_name      0
place_name       0
lg_population    0
ua               0
dtype: int64

## Cleaning the columns

In [8]:
# strip spaces from ORI
crosswalk['ori'] = crosswalk['ori'].str.strip()

In [9]:
len(crosswalk.ori.unique())

35433

In [10]:
fips_str = crosswalk['county_fips'].astype(str)
lengths = fips_str.str.len()

# 2. Build a mask for FIPS codes with length < 5
mask_short = lengths < 5

# 3. Get the unique short FIPS codes
short_fips_unique = crosswalk.loc[mask_short, 'county_fips'].unique()

print("Number of FIPS codes with < 5 digits:", len(short_fips_unique))
print("Examples of short FIPS codes:")
print(short_fips_unique[:20])  # show first 20

Number of FIPS codes with < 5 digits: 315
Examples of short FIPS codes:
[1001 1003 1005 1007 1009 1011 1013 1015 1017 1019 1021 1023 1025 1027
 1029 1031 1033 1035 1037 1039]


In [11]:
# filling the county fips code with less than 5 digits with leading zeros
crosswalk['county_fips'] = crosswalk['county_fips'].astype(str).str.zfill(5)

In [12]:
# filling the place fips code with less than 5 digits with leading zeros
crosswalk['place_fips'] = crosswalk['place_fips'].astype(str).str.zfill(5)

In [13]:
crosswalk

,ori,county_fips,place_fips,state_name,county_name,place_name,lg_population,ua
0,AL0040200,01001,03220,ALABAMA,AUTAUGA,AUTAUGAVILLE TOWN,870,-1
1,AL0040100,01001,62328,ALABAMA,AUTAUGA,PRATTVILLE CITY,33960,58600
2,AL0040300,01001,62328,ALABAMA,AUTAUGA,PRATTVILLE CITY,888888888,58600
3,AL0040000,01001,99001,ALABAMA,AUTAUGA,AUTAUGA COUNTY,54571,-1
4,AL0051100,01003,04660,ALABAMA,BALDWIN,State of Alabama,888888888,5923
...,...,...,...,...,...,...,...,...
36485,WY0220000,56043,99043,WYOMING,WASHAKIE,WASHAKIE COUNTY,8533,-1
36486,WY0230100,56045,56215,WYOMING,WESTON,NEWCASTLE CITY,3532,62191
36487,WYWHP1200,56045,56215,WYOMING,WESTON,State of Wyoming,888888888,62191
36488,WY0230200,56045,79125,WYOMING,WESTON,UPTON TOWN,1100,-1


## Dropping the local government population as it shows inconsistencies and the urban code column as well.

In [14]:
crosswalk.drop(["lg_population", "ua"], axis = 1, inplace = True)

In [15]:
crosswalk

,ori,county_fips,place_fips,state_name,county_name,place_name
0,AL0040200,01001,03220,ALABAMA,AUTAUGA,AUTAUGAVILLE TOWN
1,AL0040100,01001,62328,ALABAMA,AUTAUGA,PRATTVILLE CITY
2,AL0040300,01001,62328,ALABAMA,AUTAUGA,PRATTVILLE CITY
3,AL0040000,01001,99001,ALABAMA,AUTAUGA,AUTAUGA COUNTY
4,AL0051100,01003,04660,ALABAMA,BALDWIN,State of Alabama
...,...,...,...,...,...,...
36485,WY0220000,56043,99043,WYOMING,WASHAKIE,WASHAKIE COUNTY
36486,WY0230100,56045,56215,WYOMING,WESTON,NEWCASTLE CITY
36487,WYWHP1200,56045,56215,WYOMING,WESTON,State of Wyoming
36488,WY0230200,56045,79125,WYOMING,WESTON,UPTON TOWN


In [16]:
crosswalk.to_csv("crosswalk_panel.csv")